# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AsimaZaheer/Task1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/AsimaZaheer/Task1.git

Cloning into 'Task1'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 143 (delta 53), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 1.86 MiB | 17.29 MiB/s, done.
Resolving deltas: 100% (53/53), done.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal 1: Content age (staleness)

I will check content age because older content may be more suitable for review or refresh. This is an observable signal that can support a refresh-prioritization rule.

**Verdict: CONFIRMED**

The result will be treated as directional evidence, not proof that old content needs a refresh.

In [3]:
# Signal 1: Content age buckets

df["age_bucket"] = pd.cut(
    df["content_age_days"],
    bins=[0, 90, 180, 365, float("inf")],
    labels=["0-90 days", "91-180 days", "181-365 days", "365+ days"],
    include_lowest=True
)

age_check = (
    df["age_bucket"]
    .value_counts()
    .sort_index()
    .reset_index()
)

age_check.columns = ["age_bucket", "n"]

print(age_check)

     age_bucket      n
0     0-90 days    492
1   91-180 days  11780
2  181-365 days  11368
3     365+ days   6360


### Signal 2: Search visibility (90-day impressions)

I will check 90-day impressions because a refresh decision should consider whether a page has enough search exposure to matter. Impressions are an observable search signal.

**Verdict: CONFIRMED**

This is directional evidence only. High impressions do not prove that a page needs a refresh, but they indicate that a page has measurable search visibility.

In [5]:
# Signal 2: 90-day impressions buckets

df["impression_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[0, 100, 500, 1000, 5000, float("inf")],
    labels=[
        "1-100",
        "101-500",
        "501-1,000",
        "1,001-5,000",
        "5,000+"
    ],
    include_lowest=True
)

impression_check = (
    df["impression_bucket"]
    .value_counts()
    .sort_index()
    .reset_index()
)

impression_check.columns = ["impression_bucket", "n"]

print(impression_check)

  impression_bucket     n
0             1-100  8006
1           101-500  5279
2         501-1,000  3206
3       1,001-5,000  7359
4            5,000+  6150


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline rule: Stale + Visible Content

I will prioritize pages that are both relatively old and have meaningful search visibility.

A page receives a higher baseline score when:
- content_age_days is high, and
- impressions_90d is high.

The score combines these two observable signals into one ranking score. The reason code will identify the page as a "stale_visible_page".

The action label will be "Review for Refresh".

This is a prioritization rule, not a prediction that a refresh will improve performance.

In [6]:
# Baseline Action Score
# Higher score = older + more visible page

df["age_score"] = (df["content_age_days"] / df["content_age_days"].max()) * 50

df["visibility_score"] = (
    df["impressions_90d"] / df["impressions_90d"].max()
) * 50

df["baseline_score"] = (
    df["age_score"] + df["visibility_score"]
)

# One reason code
df["reason_code"] = "stale_visible_page"

# One action label
df["action"] = "Review for Refresh"

# Rank pages
df = df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

df["rank"] = df.index + 1

# Show top 10
df[
    [
        "rank",
        "content_id",
        "content_age_days",
        "impressions_90d",
        "baseline_score",
        "reason_code",
        "action"
    ]
].head(10)

,rank,content_id,content_age_days,impressions_90d,baseline_score,reason_code,action
0,1,content_5fe46e04994d,537,517715,97.606383,stale_visible_page,Review for Refresh
1,2,content_aaef01a50def,445,517109,89.391828,stale_visible_page,Review for Refresh
2,3,content_8c19996aa890,445,509252,88.633013,stale_visible_page,Review for Refresh
3,4,content_4c36c775b818,445,463103,84.176024,stale_visible_page,Review for Refresh
4,5,content_1a9e894be2e2,482,416180,82.924426,stale_visible_page,Review for Refresh
5,6,content_db5989a78dd3,445,345111,72.780565,stale_visible_page,Review for Refresh
6,7,content_2dba2b1f9536,299,443434,69.333164,stale_visible_page,Review for Refresh
7,8,content_9532f197bbc8,445,309192,69.311572,stale_visible_page,Review for Refresh
8,9,content_2c2606c5d176,362,347399,65.643380,stale_visible_page,Review for Refresh
9,10,content_2cb567c3c89b,153,497727,61.633424,stale_visible_page,Review for Refresh


In [7]:
# Save the ranked baseline action queue

import os

output_dir = "/content/Task1/work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = f"{output_dir}/baseline_action_score.csv"

queue_columns = [
    "rank",
    "content_id",
    "content_age_days",
    "impressions_90d",
    "baseline_score",
    "reason_code",
    "action"
]

df[queue_columns].to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(df))

Saved: /content/Task1/work/outputs/baseline_action_score.csv
Rows: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-10 Review

I reviewed the top 10 pages selected by the baseline rule. The baseline prioritizes pages that are old and have high search impressions. However, a high score does not prove that a page needs a refresh, so each recommendation should be checked by a human reviewer.

| Rank | Action | Why it is here | What would make it wrong |
|---|---|---|---|
| 1 | Review for Refresh | The page is 537 days old and has 517,715 impressions, so it has high visibility and strong refresh-review priority. | The page may still be performing well and may not need an update. |
| 2 | Review for Refresh | The page is 445 days old with 517,109 impressions, making it an old but highly visible page. | Its performance may be stable despite its age. |
| 3 | Review for Refresh | The page is 445 days old and has 509,252 impressions, so the rule gives it a high score. | High impressions do not by themselves prove that a refresh is needed. |
| 4 | Review for Refresh | The page is 445 days old with 463,103 impressions, indicating an old page with substantial visibility. | The page may still satisfy search intent and perform adequately. |
| 5 | Review for Refresh | The page is 482 days old and has 416,180 impressions, giving it strong visibility and age-based priority. | The content may still be useful and current despite its age. |
| 6 | Review for Refresh | The page is 445 days old with 345,111 impressions, so it meets the stale-and-visible rule. | Its traffic may be stable or seasonal, so refreshing it may not be necessary. |
| 7 | Review for Refresh | The page is 299 days old and has 443,434 impressions, giving it substantial visibility. | Age alone may not indicate that the content is outdated. |
| 8 | Review for Refresh | The page is 445 days old with 309,192 impressions, so it is considered stale and visible. | The page may still be performing well without changes. |
| 9 | Review for Refresh | The page is 362 days old and has 347,399 impressions, making it a visible older page. | The page may not have an actual content problem. |
| 10 | Review for Refresh | The page is 153 days old but has 497,727 impressions, so its high visibility makes it a review candidate under the rule. | It is relatively newer than the other pages, so age may not justify a refresh. |

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks

The baseline can make weak recommendations when a page is old and highly visible but does not actually have a content problem.

1. **Old but healthy page:** A page may be old and have high impressions, so the rule ranks it highly, even though it may still be performing well and remain relevant.

2. **High-volume page without evidence of decline:** High impressions show visibility, but they do not prove that the page is losing performance or needs a refresh.

3. **Seasonal content:** A page may have high impressions during a particular period, but its performance may naturally change because of seasonality. The baseline does not account for this.

These examples show that the baseline is a decision-support tool, not a guarantee that a page should be refreshed. Human review is still required before taking action.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.